In [1]:
import sys
import os

# sobe um nível (da pasta notebooks → raiz do projeto)
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)


In [2]:
from pipeline.context import ClaimContext
from pipeline.pipeline import Pipeline

from modules.llm.ollama_interface import OllamaLLM
from modules.question_generation.question_generator import QuestionGenerator
from modules.search.web_search import WebSearch
from modules.parsing.document_parser import DocumentParser
from modules.segmentation.passage_extractor import PassageExtractor
from modules.retrieval.bm25_retriever import BM25Retriever

from modules.verdict.rule_verdict import RuleVerdict
from modules.verdict.majority_verdict import MajorityVerdict
from modules.verdict.weighted_verdict import WeightedVerdict
from modules.verdict.llm_verdict import LLMVerdict

from modules.stance.llm_stance_detector import LLMStanceDetector
from modules.reranking.cross_encoder_reranker import CrossEncoderReranker
from modules.qa.qa_generator import QAGenerator
from modules.search.web_search import WebSearch
from modules.justification.justification_generator import JustificationGenerator

from dotenv import load_dotenv
import os

/home/systemp2w/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

searcher = WebSearch(api_key=os.getenv("BRAVE_API_KEY"))
llm = OllamaLLM()

In [4]:
pipeline = Pipeline(
    question_generator=QuestionGenerator(llm),
    searcher=searcher,
    parser=DocumentParser(),
    segmenter=PassageExtractor(),
    retriever=BM25Retriever(),
    qa_generator=QAGenerator(llm),
    stance_detector=LLMStanceDetector(llm),
    verdict_predictor=MajorityVerdict(),
    justification_generator=JustificationGenerator(llm),
    reranker=CrossEncoderReranker()
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 945.02it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU1 NVIDIA GeForce GTX 1050 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  queued_call()
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pyt

In [5]:
context = ClaimContext(
    claim_id=1,
    claim_text="Hunter Biden had no experience in the energy sector before Burisma."
)

In [6]:
result = pipeline.run(context)

CACHE HIT: Hunter Biden had no experience in the energy sector before Burisma.
API SEARCH: 1. What was Hunter Biden's profession from 2014 to 2020?
API SEARCH: 2. Which Ukrainian company did Hunter Biden join as a board member in 2014?
API SEARCH: 3. What was Hunter Biden's education background related to?
PAGE CACHE HIT: https://en.wikipedia.org/wiki/Hunter_Biden
PAGE CACHE HIT: https://oversight.house.gov/the-bidens-influence-peddling-timeline/


In [9]:
print("\nCLAIM:")
print(result.claim)

print("\nTOP EVIDENCE:")

for evidence in result.evidence:
    print("-", evidence)

print("\nSTANCE CLASSIFICATION:")

for evidence, stance in result.stances:
    print(f"[{stance}] {evidence}")

print("\nQA PAIRS:\n")

for qa in result.qa_pairs:

    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print()

print("\nFINAL VERDICT:")
print(result.verdict)

print("\nJUSTIFICATION:\n")
print(result.justification)


CLAIM:
Hunter Biden had no experience in the energy sector before Burisma.

TOP EVIDENCE:
- control of Ukraine's Crimean Peninsula. Burisma's owner, Mykola Zlochevskiy, a former environment and natural resources minister, was under investigation for corruption, accused of having used his position to acquire lucrative gas fields. Hunter Biden had no background in the energy industry, sparking speculation he was hired to provide political cover for Burisma and its owner. The United States and its allies were in the process of placing sanctions on former officials in Yanukovych's government who were believed to have enriched themselves. Hunter Biden was paid as much as $50,000 a month, a large amount by U.S. standards. Board members of
- anything to help Ukraine.” But as we say above, Hunter left the Burisma board back in 2019 — almost three years before the war in Ukraine began. Editor’s note: FactCheck.org is one of several organizations working with Facebook to debunk misinformation s